In [7]:

import json
import torch
import pandas as pd

from pathlib import Path
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)


BASE_DIR = Path(
    r"E:\Work\AI\MAKTAB\HW-CW-S02\NLP Project"
)

RATING_MODEL_DIR = (
    BASE_DIR
    / "digikala_rating_sampling_v1"/"digikala_parsbert_semi_supervised"
    / "final_model"
)

RECOMMENDATION_MODEL_DIR = (
    BASE_DIR
    / "parsbert_digikala_best_label_clean_KMEANS"
)

RATING_FILE = (
    BASE_DIR
    / "digikala_rating_sampling_v1"
    / "digikala_rating_semi_supervised_600k_v1.csv"
)

TEXT_COLUMN = "text"
RATING_LABEL_COLUMN = "rating_band"

MAX_LENGTH = 128

RATING_FALLBACK_LABELS = {
    0: "low",
    1: "medium",
    2: "high"
}

RECOMMENDATION_FALLBACK_LABELS = {
    0: "no_idea",
    1: "not_recommended",
    2: "recommended"
}

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# --------------------------------------------------
# 1. Check paths
# --------------------------------------------------

required_paths = [
    RATING_MODEL_DIR,
    RECOMMENDATION_MODEL_DIR,
    RATING_FILE
]

for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(
            f"Path not found: {path}"
        )


# --------------------------------------------------
# 2. Load label mapping
# --------------------------------------------------

def load_label_mapping(model_dir, fallback_labels):

    mapping_file = model_dir / "label_mapping.json"

    if mapping_file.exists():

        with open(mapping_file, "r", encoding="utf-8") as file:
            mapping = json.load(file)

        if "id2label" in mapping:

            return {
                int(key): str(value)
                for key, value in mapping["id2label"].items()
            }

        if "label2id" in mapping:

            return {
                int(value): str(key)
                for key, value in mapping["label2id"].items()
            }

    return fallback_labels


rating_labels = load_label_mapping(
    RATING_MODEL_DIR,
    RATING_FALLBACK_LABELS
)

recommendation_labels = load_label_mapping(
    RECOMMENDATION_MODEL_DIR,
    RECOMMENDATION_FALLBACK_LABELS
)


# --------------------------------------------------
# 3. Load rating CSV
# --------------------------------------------------

rating_df = pd.read_csv(
    RATING_FILE,
    encoding="utf-8-sig",
    low_memory=False
)

if TEXT_COLUMN not in rating_df.columns:
    raise ValueError(
        f"Column '{TEXT_COLUMN}' not found in rating CSV."
    )

if RATING_LABEL_COLUMN not in rating_df.columns:
    if "labels" in rating_df.columns:

        rating_df[RATING_LABEL_COLUMN] = (
            pd.to_numeric(
                rating_df["labels"],
                errors="coerce"
            ).map(RATING_FALLBACK_LABELS)
        )

    else:
        raise ValueError(
            "Neither rating_band nor labels column exists."
        )


rating_df[TEXT_COLUMN] = (
    rating_df[TEXT_COLUMN]
    .fillna("")
    .astype(str)
    .str.strip()
)

rating_df[RATING_LABEL_COLUMN] = (
    rating_df[RATING_LABEL_COLUMN]
    .astype("string")
    .str.strip()
    .str.lower()
)

rating_df = rating_df[
    rating_df[TEXT_COLUMN].ne("")
    & rating_df[RATING_LABEL_COLUMN].notna()
].copy()


rating_lookup = (
    rating_df.groupby(TEXT_COLUMN)[RATING_LABEL_COLUMN]
    .apply(lambda values: sorted(set(values.astype(str))))
    .to_dict()
)

print("\nRating CSV loaded.")
print("Labeled comments:", len(rating_df))


# --------------------------------------------------
# 4. Load Rating model
# --------------------------------------------------

print("\nLoading Rating model...")

rating_tokenizer = AutoTokenizer.from_pretrained(
    str(RATING_MODEL_DIR)
)

rating_model = AutoModelForSequenceClassification.from_pretrained(
    str(RATING_MODEL_DIR)
)

rating_model.to(device)
rating_model.eval()


# --------------------------------------------------
# 5. Load Recommendation model
# --------------------------------------------------

print("\nLoading Recommendation model...")

recommendation_tokenizer = AutoTokenizer.from_pretrained(
    str(RECOMMENDATION_MODEL_DIR)
)

recommendation_model = AutoModelForSequenceClassification.from_pretrained(
    str(RECOMMENDATION_MODEL_DIR)
)

recommendation_model.to(device)
recommendation_model.eval()

print("\nBoth models loaded successfully.")


# --------------------------------------------------
# 6. Prediction function
# --------------------------------------------------

def predict_comment(text, model, tokenizer, labels):

    encoded = tokenizer(
        text,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    encoded = {
        key: value.to(device)
        for key, value in encoded.items()
    }

    with torch.inference_mode():

        if device.type == "cuda":

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                outputs = model(**encoded)

        else:
            outputs = model(**encoded)

        probabilities = torch.softmax(
            outputs.logits.float(),
            dim=-1
        )[0]

    predicted_id = int(
        torch.argmax(probabilities).item()
    )

    predicted_label = labels.get(
        predicted_id,
        f"unknown_{predicted_id}"
    )

    confidence = float(
        probabilities[predicted_id].item()
    )

    probability_dict = {}

    for i in range(len(probabilities)):

        label = labels.get(i, f"unknown_{i}")

        probability_dict[label] = float(
            probabilities[i].item()
        )

    return predicted_label, confidence, probability_dict


# --------------------------------------------------
# 7. Inference loop
# --------------------------------------------------

while True:

    print("\n" + "=" * 65)

    comment = input(
        "Enter a comment (or type exit): "
    ).strip()

    if comment.lower() == "exit":
        print("Inference finished.")
        break

    if not comment:
        print("Please enter a valid comment.")
        continue

    print("\nComment:")
    print(comment)

    # ----------------------------------------------
    # Rating label from CSV
    # ----------------------------------------------

    known_rating = rating_lookup.get(comment)

    print("\n" + "-" * 55)
    print("RATING LABEL FROM CSV")
    print("-" * 55)

    if known_rating:

        if len(known_rating) == 1:
            print("Actual Rating:", known_rating[0])

        else:
            print("Multiple Rating labels:", known_rating)

    else:
        print("Rating label not found in CSV.")

    # ----------------------------------------------
    # Rating prediction
    # ----------------------------------------------

    rating_result, rating_confidence, rating_probs = predict_comment(
        comment,
        rating_model,
        rating_tokenizer,
        rating_labels
    )

    print("\n" + "-" * 55)
    print("RATING MODEL")
    print("-" * 55)

    print("Predicted Rating:", rating_result)

    print(
        "Confidence:",
        f"{rating_confidence * 100:.2f}%"
    )

    print("\nRating probabilities:")

    for label, probability in rating_probs.items():

        print(
            f"{label}: {probability * 100:.2f}%"
        )

    # ----------------------------------------------
    # Recommendation prediction
    # ----------------------------------------------

    recommendation_result, recommendation_confidence, recommendation_probs = predict_comment(
        comment,
        recommendation_model,
        recommendation_tokenizer,
        recommendation_labels
    )

    print("\n" + "-" * 55)
    print("RECOMMENDATION MODEL")
    print("-" * 55)

    print(
        "Predicted Recommendation:",
        recommendation_result
    )

    print(
        "Confidence:",
        f"{recommendation_confidence * 100:.2f}%"
    )

    print("\nRecommendation probabilities:")

    for label, probability in recommendation_probs.items():

        print(
            f"{label}: {probability * 100:.2f}%"
        )

    # ----------------------------------------------
    # Final result
    # ----------------------------------------------

    print("\n" + "=" * 65)
    print("FINAL RESULT")
    print("=" * 65)

    print("Rating:", rating_result)

    print(
        "Recommendation:",
        recommendation_result
    )

    print("=" * 65)


Device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU

Rating CSV loaded.
Labeled comments: 600000

Loading Rating model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Loading Recommendation model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Both models loaded successfully.


Comment:
بد نیست

-------------------------------------------------------
RATING LABEL FROM CSV
-------------------------------------------------------
Rating label not found in CSV.

-------------------------------------------------------
RATING MODEL
-------------------------------------------------------
Predicted Rating: medium
Confidence: 68.95%

Rating probabilities:
low: 22.57%
medium: 68.95%
high: 8.48%

-------------------------------------------------------
RECOMMENDATION MODEL
-------------------------------------------------------
Predicted Recommendation: no_idea
Confidence: 79.07%

Recommendation probabilities:
no_idea: 79.07%
not_recommended: 3.99%
recommended: 16.94%

FINAL RESULT
Rating: medium
Recommendation: no_idea


Comment:
خوشم اومد ازش

-------------------------------------------------------
RATING LABEL FROM CSV
-------------------------------------------------------
Rating label not found in CSV.

---------------------------